In [ ]:
!pip install -U langchain langchain-community langchain-openai beautifulsoup4 openai pydub pygame

Defaulting to user installation because normal site-packages is not writeable


In [5]:
import os
import io
from dotenv import load_dotenv
from openai import OpenAI
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import pygame # অডিও প্লে করার জন্য

# ১. এনভায়রনমেন্ট সেটআপ
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ২. ওয়েবসাইট থেকে ডাটা লোড ও প্রসেসিং
url = "https://betopiagroup.com/"
loader = WebBaseLoader(url)
data = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(data)

embeddings = OpenAIEmbeddings()
vector_db = FAISS.from_documents(chunks, embeddings)

# ৩. RAG চেইন সেটআপ
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
template = """Answer the question based only on the following context:
{context}
Question: {question}"""
prompt = ChatPromptTemplate.from_template(template)

retriever = vector_db.as_retriever()
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# --- ভয়েস ফাংশনসমূহ ---

def speak_text(text):
    """টেক্সটকে ভয়েসে রূপান্তর করে প্লে করবে"""
    response = client.audio.speech.create(
        model="tts-1",
        voice="alloy", # আপনি echo, fable, onyx, nova, shimmer ও ব্যবহার করতে পারেন
        input=text
    )
    # অডিও ফাইল প্লে করা
    byte_stream = io.BytesIO(response.content)
    pygame.mixer.init()
    pygame.mixer.music.load(byte_stream)
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        continue

# ৪. ভয়েস চ্যাট লুপ
print("AI: আমি আপনার ওয়েবসাইট রিড করেছি। এখন প্রশ্ন করুন (টাইপ করুন অথবা ভয়েস মোড চালু করুন)...")



C:\Users\parvez\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html
AI: আমি আপনার ওয়েবসাইট রিড করেছি। এখন প্রশ্ন করুন (টাইপ করুন অথবা ভয়েস মোড চালু করুন)...


In [ ]:
while True:
    # এখানে শুধু প্রম্পট থাকবে, প্রশ্ন আপনি নিজে টাইপ করবেন
    query = input("\nWhat are the core beliefs of Betopia Group?") 
    
    if query.lower() == 'exit':
        print("বিদায়!")
        break
        
    response = rag_chain.invoke(query) # আপনার টাইপ করা প্রশ্নের উত্তর খুঁজবে
    print(f"\nAI Response: {response}") # টেক্সট দেখাবে
    speak_text(response) # মুখে বলে শোনাবে

In [1]:
!pip install SpeechRecognition pyaudio

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/32.9 MB ? eta -:--:--
    --------------------------------------- 0.8/32.9 MB 8.3 MB/s eta 0:00:04
   --- ------------------------------------ 3.1/32.9 MB 10.2 MB/s eta 0:00:03
   ------- -------------------------------- 5.8/32.9 MB 10.7 MB/s eta 0:00:03
   ---------- ----------------------------- 8.4/32.9 MB 11.1 MB/s eta 0:00:03
   ------------- -------------------------- 11.0/32.9 MB 11.3 MB/s eta 0:00:02
   ---------------- ----------------------- 13.4/32.9 MB 11.5 MB/s eta 0:00:02
   ------------------- -------------------- 15.7/32.9 MB 11.4 MB/s eta 0:00:02
   ---------------------- ----------------- 18.1/32.9 MB 11.5 MB/s eta 0:00:02
   ------------------------- -------------- 20.7/32.9 MB 11.6 MB/s eta 0:00:02
   ---------------------------- ----------- 23.1/32.9 MB 11.6 MB/s eta 0:00:01
   ------------------------------ --------- 25.4/32.9 MB 11.6 MB/s 

In [ ]:
import speech_recognition as sr

def listen_to_user():
    """আপনার কথা শুনে সেটাকে টেক্সটে রূপান্তর করবে"""
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        print("\n--- এখন আপনার প্রশ্নটি বলুন (শুনছি...) ---")
        recognizer.adjust_for_ambient_noise(source) # আসেপাশের নয়েজ অ্যাডজাস্ট করবে
        audio = recognizer.listen(source)

    try:
        # কথাকে টেক্সটে রূপান্তর (Google API ব্যবহার করে)
        user_text = recognizer.recognize_google(audio)
        print(f"আপনি বলেছেন: {user_text}")
        return user_text
    except sr.UnknownValueError:
        print("দুঃখিত, আমি আপনার কথা বুঝতে পারিনি।")
        return None
    except sr.RequestError:
        print("ইন্টারনেট কানেকশনে সমস্যা হচ্ছে।")
        return None

# --- মেইন লুপ ---
print("AI: আমি আপনার ওয়েবসাইট রিড করেছি। এখন প্রশ্ন করুন (ভয়েস মোড চালু)...")



AI: আমি আপনার ওয়েবসাইট রিড করেছি। এখন প্রশ্ন করুন (ভয়েস মোড চালু)...

--- এখন আপনার প্রশ্নটি বলুন (শুনছি...) ---
দুঃখিত, আমি আপনার কথা বুঝতে পারিনি।

--- এখন আপনার প্রশ্নটি বলুন (শুনছি...) ---
দুঃখিত, আমি আপনার কথা বুঝতে পারিনি।

--- এখন আপনার প্রশ্নটি বলুন (শুনছি...) ---
আপনি বলেছেন: hello can you hear me


NameError: name 'rag_chain' is not defined

In [3]:
while True:
    # ১. আপনার কথা শুনবে
    query = listen_to_user()
    
    if query:
        if query.lower() == 'exit' or query.lower() == 'quit':
            print("বিদায়! ভালো থাকবেন।")
            break
            
        # ২. ওয়েবসাইট থেকে উত্তর খুঁজবে (RAG)
        response = rag_chain.invoke(query)
        
        # ৩. স্ক্রিনে উত্তর দেখাবে
        print(f"AI Response: {response}")
        
        # ৪. মুখে উত্তর বলে শোনাবে
        speak_text(response)


--- এখন আপনার প্রশ্নটি বলুন (শুনছি...) ---
দুঃখিত, আমি আপনার কথা বুঝতে পারিনি।

--- এখন আপনার প্রশ্নটি বলুন (শুনছি...) ---
দুঃখিত, আমি আপনার কথা বুঝতে পারিনি।

--- এখন আপনার প্রশ্নটি বলুন (শুনছি...) ---
দুঃখিত, আমি আপনার কথা বুঝতে পারিনি।

--- এখন আপনার প্রশ্নটি বলুন (শুনছি...) ---
আপনি বলেছেন: hello can you hear me


NameError: name 'rag_chain' is not defined